## NanoGPT

https://github.com/karpathy/nanoGPT/blob/master/train.py

In [ ]:
"""
This is the core architecture file for nanoGPT. It defines the entire
Generative Pre-trained Transformer (GPT) model from scratch.

If you understand standard Transformers, this is essentially the "Decoder"
stack, built to predict the next token.
"""

import math
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

# -----------------------------------------------------------------------------
# 1. Utility: Layer Normalization
# -----------------------------------------------------------------------------
class LayerNorm(nn.Module):
    """
    Standard Layer Normalization, but customized to allow turning off the 'bias'.
    Why? In newer architectures, removing biases can sometimes make the model
    train slightly faster and perform a bit better.
    """
    def __init__(self, ndim, bias):
        super().__init__()
        # 'Weight' scales the normalized output, 'bias' shifts it.
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

# -----------------------------------------------------------------------------
# 2. The Heart of the Transformer: Causal Self-Attention
# -----------------------------------------------------------------------------
class CausalSelfAttention(nn.Module):
    """
    This is where tokens "look" at other tokens to understand context.
    'Causal' means a token can only look at *past* tokens, not future ones
    (otherwise, it could cheat when predicting the next word).
    """
    def __init__(self, config):
        super().__init__()
        # Ensure our embedding size divides evenly into our number of heads
        assert config.n_embd % config.n_head == 0

        # Instead of 3 separate linear layers for Query, Key, and Value,
        # we do one big projection and split it later. It's more efficient.
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        
        # This projects the attention output back to the original embedding size
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        
        # Regularization to prevent overfitting
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

        # Flash Attention is a highly optimized algorithm for computing attention.
        # It's mathematically identical but runs much faster on modern GPUs.
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')

        # The "mask" that hides future tokens. It's a lower-triangular matrix (tril).
        # We register it as a "buffer" so PyTorch knows it's part of the model state,
        # but it isn't a parameter that gets updated during training.
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        # B = Batch size, T = Sequence length (Time), C = Embedding dimension
        B, T, C = x.size() 

        # 1. Calculate Query, Key, and Value
        # Pass input through our single linear layer, then split it into 3 equal chunks
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # Reshape to separate out the different attention "heads".
        # We transpose so the sequence length (T) and heads are in the right order for math.
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, heads, T, head_size)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) 
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) 

        # 2. Compute Attention
        if self.flash:
            # The fast, optimized PyTorch 2.0+ way
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # The manual math way (good for understanding the theory)
            # a. Calculate attention scores: Query dot Key
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            
            # b. Apply the causal mask: Replace upper triangle (future tokens) with -infinity
            # so they become 0 after softmax.
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            
            # c. Turn scores into probabilities
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            
            # d. Multiply probabilities by Values to get the final output
            y = att @ v 

        # 3. Re-assemble the heads
        # Put the heads back together and reshape to original (Batch, Time, Channels)
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # 4. Final output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

# -----------------------------------------------------------------------------
# 3. The Multi-Layer Perceptron (MLP)
# -----------------------------------------------------------------------------
class MLP(nn.Module):
    """
    After attention figures out how tokens relate to each other, the MLP 
    processes the information *within* each token independently.
    It expands the dimension by 4x, applies an activation, and shrinks it back.
    """
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        # GELU is a smoother version of ReLU, standard in GPT models
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

# -----------------------------------------------------------------------------
# 4. A Single Transformer Block
# -----------------------------------------------------------------------------
class Block(nn.Module):
    """
    One layer of the Transformer. It combines Attention and the MLP.
    A full GPT model is basically just many of these blocks stacked on top of each other.
    """
    def __init__(self, config):
        super().__init__()
        # Pre-attention normalization
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        # Pre-MLP normalization
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        # Notice the residual connections: "x + layer(x)"
        # This helps gradients flow during training and prevents the vanishing gradient problem.
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

# -----------------------------------------------------------------------------
# 5. Model Configuration
# -----------------------------------------------------------------------------
@dataclass
class GPTConfig:
    """
    A simple class to hold the hyperparameters that define the model's size and shape.
    """
    block_size: int = 1024     # Max sequence length (context window)
    vocab_size: int = 50304    # Number of unique tokens in the vocabulary
    n_layer: int = 12          # Number of Transformer blocks
    n_head: int = 12           # Number of attention heads
    n_embd: int = 768          # Embedding dimension size
    dropout: float = 0.0       # Dropout probability
    bias: bool = True          # Whether to use bias in linear layers

# -----------------------------------------------------------------------------
# 6. The Full GPT Model
# -----------------------------------------------------------------------------
class GPT(nn.Module):
    """
    The main container that holds everything together.
    """
    def __init__(self, config):
        super().__init__()
        self.config = config

        # The core Transformer components stored in a dictionary
        self.transformer = nn.ModuleDict(dict(
            # Token Embedding: Turns token IDs (integers) into dense vectors
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            # Position Embedding: Tells the model *where* in the sequence a token is
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            
            # The stack of Transformer blocks
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            
            # Final LayerNorm before the output
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        
        # The final layer that turns embeddings back into vocabulary probabilities
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight Tying: We use the exact same weights for the initial token embedding
        # and the final output layer. This saves a massive amount of memory and often improves learning.
        self.transformer.wte.weight = self.lm_head.weight

        # Initialize weights (standard practice)
        self.apply(self._init_weights)
        
        # Print total parameter count so we know how big the model is
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    # [Helper initialization methods omitted for brevity in demo]
    # ... def get_num_params(...), def _init_weights(...) ...

    def forward(self, idx, targets=None):
        """
        The forward pass.
        'idx' are the input token IDs. 'targets' are the correct next tokens (used during training).
        """
        device = idx.device
        b, t = idx.size() # Batch size, Sequence length
        
        # Create an array of positions: [0, 1, 2, ..., t-1]
        pos = torch.arange(0, t, dtype=torch.long, device=device) 

        # 1. Get embeddings
        tok_emb = self.transformer.wte(idx) # Token embeddings
        pos_emb = self.transformer.wpe(pos) # Position embeddings
        
        # Add them together (Token Meaning + Token Position)
        x = self.transformer.drop(tok_emb + pos_emb)

        # 2. Pass through all Transformer blocks
        for block in self.transformer.h:
            x = block(x)
            
        # 3. Final normalization
        x = self.transformer.ln_f(x)

        # 4. Calculate loss (if training) or just output logits (if generating)
        if targets is not None:
            # We have the answers, so calculate how wrong the model was
            logits = self.lm_head(x)
            # Cross Entropy compares the predicted probabilities (logits) against the actual targets
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # Inference mode (generating text). 
            # We only care about predicting the very *next* token, so we only process the last position in the sequence.
            logits = self.lm_head(x[:, [-1], :]) 
            loss = None

        return logits, loss